## Using **Tweet Generator** Example Here

In [11]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
import os
from pydantic import BaseModel, Field
from dotenv import load_dotenv

In [5]:
load_dotenv()

True

In [6]:
generator_llm = ChatGroq(
    model="openai/gpt-oss-120b",   
    api_key=os.getenv("GROQ_API_KEY"),
)

evaluator_llm = ChatGroq(
    model="openai/gpt-oss-20b",   
    api_key=os.getenv("GROQ_API_KEY"),
)

optimizer_llm = ChatGroq(
    model="openai/gpt-oss-120b",   
    api_key=os.getenv("GROQ_API_KEY"),
)

In [ ]:
class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [13]:
structured_evaluator_llm = evaluator_llm. with_structured_output(TweetEvaluation)

In [8]:
# state 
class TweetState(TypedDict):
    topic: str 
    tweet: str 
    evaluation: Literal["approved", "needs_improvement"]
    feedback: str 
    iteration: int 
    max_iteration: int 

In [9]:
def generate_tweet(state: TweetState):
    #prompt
    messages = [
            SystemMessage(content="You are a funny and clever Twitter/X influencer."),
            HumanMessage(content=f"""
    Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

    Rules:
    - Do NOT use question-answer format.
    - Max 280 characters.
    - Use observational humor, irony, sarcasm, or cultural references.
    - Think in meme logic, punchlines, or relatable takes.
    - Use simple, day to day english
    """)
    ]
    # generator llm
    response = generator_llm.invoke(messages)

    return {'tweet':response}

In [15]:
def evaluate_tweet(state: TweetState):
    # prompt 
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
    Evaluate the following tweet:

    Tweet: "{state['tweet']}"

    Use the criteria below to evaluate the tweet:

    1. Originality – Is this fresh, or have you seen it a hundred times before?  
    2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
    3. Punchiness – Is it short, sharp, and scroll-stopping?  
    4. Virality Potential – Would people retweet or share it?  
    5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

    Auto-reject if:
    - It's written in question-answer format (e.g., "Why did..." or "What happens when...")
    - It exceeds 280 characters
    - It reads like a traditional setup-punchline joke
    - Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

    ### Respond ONLY in structured format:
    - evaluation: "approved" or "needs_improvement"  
    - feedback: One paragraph explaining the strengths and weaknesses 
    """)
    ]

    response = structured_evaluator_llm.invoke(messages)

    return {'evaluation': response.evaluation, 'feedback':response.feedback}


In [16]:
def optimize_tweet(state: StateGraph):

    # promot
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
        Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

        Rules:
        - Do NOT use question-answer format.
        - Max 280 characters.
        - Use observational humor, irony, sarcasm, or cultural references.
        - Think in meme logic, punchlines, or relatable takes.
        - Use simple, day to day english
        """)
        ]

    response = optimizer_llm.invoke(messages).content
    iteration = state['iteration'] + 1

    return {'tweet' : response, 'iteration' : iteration}

In [ ]:
# graph
graph = StateGraph(TweetState)

# add nodes
graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)

# add edges
graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')